# Native three-dimensional Stokes, Brinkman and Oseen

Tetrahedral MHM uses continuous local velocity/pressure fields and independent vector traces on macrofaces. This notebook checks an affine patch and replays the four manufactured convergence studies; these are original 3D validation cases, not historical article meshes.

The application declares P2/P1 Taylor–Hood local equations with existing
physical tetrahedral tabulation and quadrature:

$$
\begin{aligned}
a_T((u,p),(v,q))&=(\nabla u,\nabla v)_T
 -(p,\operatorname{div}v)_T-(q,\operatorname{div}u)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\qquad
C_T(u,p)=-\langle u,\eta\rangle_{\partial T}.
\end{aligned}
$$

Each local partition uses dyadic refinement two and three declared translation
modes with physical velocity-integral moments. The P1 Bernstein face modes are
interleaved in three Cartesian components with the fixed normal orientation.
`Equation` explicitly supplies negative velocity boundary moments, and an
integral gauge fixes physical pressure at zero mean. The multiplier represents
negative grad-grad pseudotraction, not symmetric Cauchy traction.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/034cbfefe5c419cb85babc744837dfa38f9bd6ac45bd08194fe5c54044d548f6/54_flow3d-companion.zip"
COMPANION_SHA256 = "034cbfefe5c419cb85babc744837dfa38f9bd6ac45bd08194fe5c54044d548f6"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/54_flow3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.tetrahedron import TetraMesh

from functools import partial
from pymhm import Equation, MultiscaleProblem, assemble
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from examples.formulations.tetrahedral_vector import (
    TetrahedralVectorSpace, vector_boundary_moments,
    tetra_velocity_pressure_equations, tetra_velocity_fields,
    tetra_displacement_pressure_equations, tetra_displacement_fields,
)


In [ ]:
def velocity(x):
    """Divergence-free affine field with inhomogeneous boundary data."""
    return np.column_stack((x[:, 0], -x[:, 1], np.ones(len(x))))

def pressure(x):
    """Zero-mean affine pressure on the unit cube."""
    return x[:, 0] + x[:, 1] - 1

mesh = TetraMesh.unit_cube()
skeleton = TriangularSkeleton(mesh, degree=1)
local_space = TetrahedralVectorSpace(mesh, skeleton, 2, 1, 2, 5)
provider = partial(tetra_velocity_pressure_equations, data=local_space, source=(1.0, 1.0, 0.0))
boundary, fixed = vector_boundary_moments(skeleton, velocity, {}, 5)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(3 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), 3 * skeleton.size, (3,) * len(mesh.cells), fixed=fixed,
)
with threadpool_limits(1):
    system = assemble(problem)
    gauge = system.mean_constraint([record[2] for record in system.local_metadata], 0.0)
    coefficients = system.solve(constraints=(gauge,))
    patch = tetra_velocity_fields(system, coefficients, local_space)
    errors = {"velocity": patch.l2_error(velocity),
              "pressure": patch.pressure_l2_error(pressure),
              "divergence": patch.divergence_l2()}
assert max(errors.values()) < 1e-10
errors


The pressure mean is a gauge because every boundary component prescribes velocity. Pseudotraction uses the grad-grad convention; the raw stress is not an H(div) reconstruction.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    folder = ROOT / "examples/results/flow3d"
    campaign = json.loads((folder / "campaign.json").read_text())
    rows = campaign["rows"]
    assert len(rows) == 20
    for row in rows:
        assert hashlib.sha256((folder / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
        assert row["max_error_quadrature_relative_difference"] < 1e-8
        assert row["backward_residual"] < 1e-8
    for case in sorted({row["case"] for row in rows}):
        selected = sorted((row for row in rows if row["case"] == case),
                          key=lambda row: row["macro_subdivisions"])
        assert [row["macro_subdivisions"] for row in selected] == [1, 2, 3, 4, 5]
        assert all(selected[-1][key] < selected[0][key]
                   for key in ("velocity_l2", "pressure_l2", "velocity_h1_seminorm"))
    [(r["case"], r["macro_subdivisions"], r["velocity_l2"], r["pressure_l2"])
     for r in rows]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    figures = ROOT / "docs/figures/flow3d"
    for name in ["convergence", "stokes-th2-fields", "brinkman-usfem1-fields",
                 "brinkman-usfem2-fields", "oseen-p2-fields", "oseen-p2-components"]:
        display(Image(filename=str(figures / f"{name}.png")))


The slice panels retain one-sided fine-element values and the true macro boundaries. Error quadrature orders seven and eight are checked independently. The reported time is elapsed campaign time under concurrent activity, not a speedup measurement. This evidence covers the stated finite spaces and coefficients; it is not a uniform inf-sup or coefficient-robustness certificate.